# Gradient Boosting: Fitting the Residuals, From Scratch to XGBoost

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/gradient_boosting_residuals.ipynb)

Companion notebook to [Gradient Boosting: Fitting the Residuals, From Scratch to XGBoost](https://sesen.ai/blog/gradient-boosting-residuals-to-xgboost).

We build gradient boosting from scratch, watch it fit residuals round by round, see it overfit (and how shrinkage helps), then compare against XGBoost's regularised second-order objective.

## Setup

In [ ]:
# Uncomment on Colab
# !pip install scikit-learn xgboost matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
rng = np.random.default_rng(0)

## A wiggly 1D target

Stumps (depth-1 trees) cannot represent this curve alone. Boosting builds it from their sum.

In [ ]:
def target(x): return np.sin(2*x) + 0.5*np.sin(5*x) + 0.15*x
N = 120
X = np.sort(rng.uniform(0, 6, N)).reshape(-1, 1)
y = target(X.ravel()) + rng.normal(0, 0.18, N)
Xg = np.linspace(0, 6, 400).reshape(-1, 1); ytrue = target(Xg.ravel())
plt.scatter(X, y, s=14, alpha=.6); plt.plot(Xg, ytrue, 'k--'); plt.title('data + true function'); plt.show()

## Gradient boosting in twelve lines

Start from the mean, compute residuals, fit a tree to them, add a shrunk version, repeat. For squared loss the residual **is** the negative gradient.

In [ ]:
class GradientBoost:
    def __init__(self, n_rounds=60, lr=0.3, max_depth=1):
        self.n_rounds, self.lr, self.max_depth = n_rounds, lr, max_depth
    def fit(self, X, y):
        self.f0 = float(np.mean(y)); F = np.full(len(y), self.f0); self.trees = []
        for m in range(self.n_rounds):
            residual = y - F                              # negative gradient of squared loss
            t = DecisionTreeRegressor(max_depth=self.max_depth, random_state=m).fit(X, residual)
            F = F + self.lr * t.predict(X); self.trees.append(t)
        return self
    def staged_predict(self, X):
        F = np.full(len(X), self.f0); out = []
        for t in self.trees:
            F = F + self.lr * t.predict(X); out.append(F.copy())
        return out
    def predict(self, X): return self.staged_predict(X)[-1]

## Watch the residuals shrink

Top: the ensemble prediction after m stumps. Bottom: the residuals the next stump will fit. The errors collapse toward zero.

In [ ]:
gb = GradientBoost(n_rounds=60, lr=0.3, max_depth=1).fit(X, y)
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for col, m in enumerate([1, 3, 10, 60]):
    F = gb.staged_predict(Xg)[m-1]
    Fd = gb.staged_predict(X)[m-1]
    axes[0, col].scatter(X, y, s=10, alpha=.5); axes[0, col].plot(Xg, ytrue, 'k--')
    axes[0, col].plot(Xg, F, 'r', lw=2); axes[0, col].set_title(f'{m} stump(s)'); axes[0, col].set_ylim(-2, 2.2)
    axes[1, col].axhline(0, color='gray'); axes[1, col].scatter(X, y - Fd, s=10, color='C0', alpha=.6)
    axes[1, col].set_ylim(-1.3, 1.3); axes[1, col].set_ylabel('residual')
plt.tight_layout(); plt.show()

## Boosting overfits: train vs test, and the role of shrinkage

Training error falls forever; test error U-turns. A smaller learning rate is slower but reaches a lower minimum.

In [ ]:
Ntr = 80
Xtr = np.sort(rng.uniform(0,6,Ntr)).reshape(-1,1); ytr = target(Xtr.ravel()) + rng.normal(0,0.35,Ntr)
Xte = np.sort(rng.uniform(0,6,400)).reshape(-1,1); yte = target(Xte.ravel()) + rng.normal(0,0.35,400)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
for lr, c in [(1.0,'C3'),(0.3,'C0'),(0.1,'C2')]:
    g = GradientBoost(n_rounds=400, lr=lr, max_depth=2).fit(Xtr, ytr)
    te = [np.mean((yte-p)**2) for p in g.staged_predict(Xte)]
    a2.plot(range(1,401), te, color=c, label=f'lr={lr} (best {min(te):.3f})')
    if lr==0.3:
        tr = [np.mean((ytr-p)**2) for p in g.staged_predict(Xtr)]
        a1.plot(range(1,401), tr, 'C0', label='train'); a1.plot(range(1,401), te, 'C3', label='test')
        a1.axvline(np.argmin(te)+1, ls='--', color='k'); a1.legend(); a1.set_title('lr=0.3: test U-turns'); a1.set_ylim(0,.6)
a2.legend(); a2.set_title('shrinkage'); a2.set_ylim(.1,.45); plt.tight_layout(); plt.show()

## XGBoost: the same loop, a regularised second-order objective

XGBoost keeps residual fitting but minimises a regularised objective using the gradient **and** Hessian. On Friedman1 it edges first-order boosting at matched settings; its real wins come at scale.

In [ ]:
import xgboost as xgb
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split
Xf, yf = make_friedman1(n_samples=2000, n_features=10, noise=1.0, random_state=0)
Xa, Xb, ya, yb = train_test_split(Xf, yf, test_size=0.4, random_state=0)

# first-order from scratch
f0 = ya.mean(); Fa = np.full(len(ya), f0); Fb = np.full(len(yb), f0); scratch = []
for m in range(300):
    t = DecisionTreeRegressor(max_depth=3, random_state=m).fit(Xa, ya - Fa)
    Fa = Fa + 0.1*t.predict(Xa); Fb = Fb + 0.1*t.predict(Xb)
    scratch.append(np.sqrt(np.mean((yb - Fb)**2)))

ev = {}
xgb.train({'max_depth':3,'eta':0.1,'objective':'reg:squarederror','lambda':1.0},
          xgb.DMatrix(Xa, label=ya), num_boost_round=300,
          evals=[(xgb.DMatrix(Xb, label=yb),'test')], evals_result=ev, verbose_eval=False)
xgr = ev['test']['rmse']
print(f'scratch best RMSE {min(scratch):.4f} | XGBoost best RMSE {min(xgr):.4f}')
plt.plot(scratch, label=f'first-order GB ({min(scratch):.3f})')
plt.plot(xgr, label=f'XGBoost ({min(xgr):.3f})')
plt.xlabel('rounds'); plt.ylabel('test RMSE'); plt.legend(); plt.title('Friedman1'); plt.show()

## What to remember

- Gradient boosting fits each weak learner to the **residuals** = negative gradient of the loss, then adds it on with a small learning rate.
- It is **gradient descent in function space** (Friedman 2001), so it generalises to any differentiable loss.
- It **overfits** if run too long: use a small learning rate plus early stopping.
- **XGBoost** keeps the loop but adds regularisation and a second-order (Hessian) objective.

## Exercises
1. **Change the loss.** Fit trees to `sign(residual)` (absolute-error gradient) and compare robustness to a few injected outliers.
2. **Classification.** Implement logistic-loss boosting: the negative gradient is `y - sigmoid(F)`. Plot the decision boundary on `make_moons`.
3. **Stochastic gradient boosting.** Subsample 50% of rows each round (Friedman 2002). Does it regularise?
4. **Tune XGBoost.** Sweep `lambda`, `gamma`, and `max_depth`. When does the gap over first-order boosting widen?
5. **Early stopping.** Add a validation split and stop when test error stops improving; compare to running all 400 rounds.